In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..', '..', '..', '..'))
sys.path.insert(0, module_path)

### Подготовка
1. Загрузим данные
2. Поделим их на train и test

In [2]:

from brs.models.rate_prediction.baseline.model import BaselineRateModel
from brs.ingestion import LoadLocalDataset

# Подгружаем сырые данные
lld = LoadLocalDataset()
data_books = lld.load_dataset("books")
data_users = lld.load_dataset("users")
data_ratings = lld.load_dataset("ratings")

# Делим на train и test
num_ratings = data_ratings.shape[0]
split = round(num_ratings * 0.9)
train, test = data_ratings[:split], data_ratings[split:]

d:\z\Conspects\_Summer\BRS\brs\ingestion\load_local.py:42: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(dataset_path)


In [3]:
from brs.preprocessing import UsersClarifyStrategy, BooksClarifyStrategy

data_users = UsersClarifyStrategy(ratings_df=data_ratings).preprocess(data_users)
data_books = BooksClarifyStrategy(
    ratings_df=data_ratings, 
    clear_users_df=data_users
).preprocess(data_books)

In [4]:
dut = data_users.copy()
dbt = data_books.copy()

In [5]:
data_users = dut.copy()
data_books = dbt.copy()

In [6]:
data_users.dtypes

User-ID                    int64
Location          string[python]
Age                        Int64
Age-Group               category
Country                   object
Mean-Rating              float64
Number-Ratings             Int64
dtype: object

In [7]:
import pandas as pd
data_users.drop(["Location", "Age"], axis=1, inplace=True)
data_users["Age-Group"] = data_users["Age-Group"].cat.add_categories(["Unknown"])
data_users["Age-Group"] = data_users["Age-Group"].fillna("Unknown")
data_users = pd.get_dummies(data_users, columns=["Age-Group"], prefix="Age-Group", drop_first=True)
data_users["Country"] = data_users["Country"].fillna("Unknown")
data_users = pd.get_dummies(data_users, columns=["Country"], prefix="Country", drop_first=True)

In [8]:
data_users.shape

(102715, 534)

In [9]:
def top_dummies(df: pd.DataFrame, col: str, top_n: int = 100, prefix: str = None) -> pd.DataFrame:
    prefix = prefix or col
    top = df[col].value_counts().nlargest(top_n).index
    df[col] = df[col].where(df[col].isin(top), other="Other")
    return pd.get_dummies(df, columns=[col], prefix=prefix, drop_first=True)


In [10]:
data_books.drop(["Year-Of-Publication", "Book-Title", "Same-Book"], axis=1, inplace=True)
data_books["Book-Author"] = data_books["Book-Author"].fillna("Unknown")
data_books["Publisher"] = data_books["Publisher"].fillna("Unknown")
data_books = top_dummies(data_books, "Book-Author", top_n=100, prefix="Book-Author")
data_books = top_dummies(data_books, "Publisher", top_n=100, prefix="Publisher")


cols = [
    "Popularity-Edition",
    "Popularity-Edition-Log",
    "Popularity-Book",
    "Popularity-Book-Log",
    "Rating-Edition",
    "Rating-Book",
]
for col in cols:
    data_books[col] = data_books[col].fillna(-1)

In [11]:
data_books.head()

,ISBN,Elderity,Elderity_log,Popularity-Edition,Popularity-Edition-Log,Popularity-Book,Popularity-Book-Log,Rating-Edition,Rating-Book,Book-Author_Alan Dean Foster,...,Publisher_Troll Communications,Publisher_Viking Books,Publisher_Vintage Books USA,Publisher_W. W. Norton &amp; Company,Publisher_Warner Books,Publisher_William Morrow &amp; Company,Publisher_Wizards of the Coast,Publisher_Worldwide Library,Publisher_Yearling Books,Publisher_Zebra Books
0,[0590567330],24,3.218876,1,0.693147,1,0.693147,0.0,0.0,False,...,False,False,False,False,False,False,False,False,False,False
1,[0964147726],29,3.401197,0,0.000000,0,0.000000,-1.0,-1.0,False,...,False,False,False,False,False,False,False,False,False,False
2,[0942320093],39,3.688879,0,0.000000,0,0.000000,-1.0,-1.0,False,...,False,False,False,False,False,False,False,False,False,False
3,[0310232546],22,3.135494,1,0.693147,1,0.693147,8.0,8.0,False,...,False,False,False,False,False,False,False,False,False,False
4,[0962295701],34,3.555348,0,0.000000,0,0.000000,-1.0,-1.0,False,...,False,False,False,False,False,False,False,False,False,False


### Pipeline/preprocess

- Считаем среднюю оценку для каждой книги на train
- Предсказываем каждому пользователю на test именно её

In [4]:
from brs.models.rate_prediction.lin_reg.pipeline import LinRegPipeline

pipeline = LinRegPipeline()

pipeline.fit(data_users, train, data_books)
pipeline.transform(test)

d:\z\Conspects\_Summer\BRS\brs\models\rate_prediction\lin_reg\pipeline.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.data_users["Age-Group"].fillna("Unknown", inplace=True)


TypeError: Cannot setitem on a Categorical with a new category (Unknown), set the categories first